# TopoDiffusionNet en miniature — entraînement

Produit les checkpoints utilisés par `demo.ipynb` :

| checkpoint | contenu |
|---|---|
| `ckpt/admt_15k.pt` | ADM-T, L_simple seule, 15 000 pas |
| `ckpt/tdn_9k.pt` | TDN : ADM-T 15k fine-tuné 9 000 pas avec L_simple + λ L_top, λ = 1e-5 |
| `ckpt/tdn_lam1e-3_9k.pt` | idem avec λ = 1e-3 (section 4 bis) |
| `ckpt/admt_suite_24k.pt` | ADM-T 15k continué sans L_top jusqu'à 24 000 pas (même budget que TDN) |
| `ckpt/abl/<variante>/last.pt` | ablations : TDN avec un seul réglage modifié (section 6, évaluées dans `ablations_limites.ipynb`) |

**Nécessite un GPU** (Colab T4 par exemple) : sans GPU, les cellules d'entraînement ne font rien.

En cas de déconnexion de Colab : relancer les sections 0 et 1, puis **la même** cellule d'entraînement — `train.py` retrouve `last.pt` dans le dossier de sortie et reprend où il en était.

## 0. Installation
Sur Colab : installe les dépendances, monte Google Drive et se place dans le dossier du projet. En local, la cellule ne fait rien.

In [ ]:
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q diffusers cripser
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/topodiffusion-mini"
    # si le code est dans un sous-dossier du même nom (comme sur le PC)
    if not os.path.exists(os.path.join(PROJECT_DIR, "data.py")):
        PROJECT_DIR = os.path.join(PROJECT_DIR, "topodiffusion-mini")
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)  # pour `from data import ...`
print("Colab" if IN_COLAB else "local", "|", os.getcwd())

In [ ]:
import shutil
import subprocess

import torch

# CKPT_DIR : checkpoints dans le dossier du projet (donc sur Drive sous Colab)
from config import ABLATIONS, CKPT_DIR, TDN_ARGS
from data import MaskDataset
from train import ckpt_step

HAS_GPU = torch.cuda.is_available()
# le Python du kernel, pour lancer les scripts (en local, `python` pourrait en désigner un autre)
PYTHON = sys.executable
# données sur le disque local de Colab (lire 10 000 PNG sur Drive est très lent)
DATA_DIR = "/content/data/shapes" if IN_COLAB else "data/shapes"
print("GPU :", torch.cuda.get_device_name(0) if HAS_GPU else "aucun (CPU) : l'entraînement sera sauté")


def run(*args, gpu_only=False):
    """Lance `python <args>` en affichant sa sortie au fil de l'eau."""
    if gpu_only and not HAS_GPU:
        print("pas de GPU : sauté")
        return
    cmd = [PYTHON, "-u", *map(str, args)]  # -u : sortie non bufferisée, affichée au fil de l'eau
    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"échec (code {proc.returncode})")


def freeze(src, dst):
    """Copie figée d'un checkpoint (last.pt est réécrit si on relance) ; n'écrase jamais."""
    if os.path.exists(dst):
        print(f"{dst} existe déjà (pas {ckpt_step(dst)}), rien n'est écrasé")
    else:
        shutil.copy(src, dst)
        print(f"{src} (pas {ckpt_step(src)}) copié dans {dst}")

## 1. Données
Sous Colab, on génère les images directement sur le disque de la machine (environ une minute) : copier 10 000 petits fichiers depuis Drive est beaucoup plus lent, et lire depuis Drive pendant l'entraînement aussi. Même graine -> mêmes images partout. En local, `data/shapes` existe déjà : rien n'est régénéré.

In [ ]:
N_IMAGES = 10_000
if os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) < N_IMAGES:
    shutil.rmtree(DATA_DIR)  # dossier incomplet (copie ou génération interrompue)
if not os.path.isdir(DATA_DIR):
    run("data.py", "--out", DATA_DIR, "--n", N_IMAGES, "--seed", 12)

ds = MaskDataset(DATA_DIR)
x, c = ds[0]
print(f"{len(ds)} images ; x : {tuple(x.shape)}, valeurs {x.unique().tolist()} ; c = {c}")

## 2. Essai court
300 pas pour vérifier que tout tourne et que `L_simple` baisse (elle part d'environ 1).

In [ ]:
run("train.py", "--data", DATA_DIR, "--out", f"{CKPT_DIR}/test", "--steps", 300, gpu_only=True)

## 3. ADM-T (L_simple seule), 15 000 pas

In [ ]:
run("train.py", "--data", DATA_DIR, "--out", f"{CKPT_DIR}/admt", "--steps", 15_000, gpu_only=True)

In [ ]:
freeze(f"{CKPT_DIR}/admt/last.pt", f"{CKPT_DIR}/admt_15k.pt")

## 4. TDN : fine-tuning d'ADM-T 15k avec L_top, 9 000 pas
`--resume` ne charge que les poids : le compteur de pas repart de 0.

In [ ]:
run("train.py", "--data", DATA_DIR, "--out", f"{CKPT_DIR}/tdn", "--resume", f"{CKPT_DIR}/admt_15k.pt",
    *TDN_ARGS, gpu_only=True)

In [ ]:
# le nom contient le pas atteint
src = f"{CKPT_DIR}/tdn/last.pt"
freeze(src, f"{CKPT_DIR}/tdn_{ckpt_step(src) // 1000}k.pt")

## 4 bis. TDN avec λ = 1e-3
Même protocole que le TDN de la section 4 (depuis ADM-T 15k, 9 000 pas, L_top pour t < 500), seul λ change : à λ = 1e-5, L_top ne pesait qu'environ 0,2 % du gradient de L_simple ; à 1e-3, environ 20 %.

En cas de coupure, relancer la même cellule : l'entraînement reprend au dernier checkpoint (tous les 500 pas). La copie figée `tdn_lam1e-3_9k.pt` est celle qu'évalue `demo.ipynb`.

In [ ]:
OUT = f"{CKPT_DIR}/abl/lam1e-3"
assert os.path.exists(f"{CKPT_DIR}/admt_15k.pt"), "admt_15k.pt introuvable"
run("train.py", "--data", DATA_DIR, "--out", OUT, "--resume", f"{CKPT_DIR}/admt_15k.pt",
    "--use_topo", "--lam", 1e-3, "--t_max_topo", 500, "--steps", 9_000, "--save_every", 500,
    gpu_only=True)
print("λ = 1e-3 : pas", ckpt_step(f"{OUT}/last.pt"))

In [ ]:
# copie figée (évaluée par demo.ipynb), puis attendre que tout soit envoyé sur Drive
freeze(f"{OUT}/last.pt", f"{CKPT_DIR}/tdn_lam1e-3_9k.pt")
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive à jour")

## 5. Baseline « même budget » : ADM-T 15k continué sans L_top
On part d'une copie complète d'`admt_15k.pt` (poids, optimiseur, pas) : `train.py` reprend donc au pas 15 000. On va jusqu'à 24 000 pas, soit les mêmes 9 000 pas supplémentaires que TDN.

In [ ]:
os.makedirs(f"{CKPT_DIR}/admt_suite", exist_ok=True)
if not os.path.exists(f"{CKPT_DIR}/admt_suite/last.pt"):
    shutil.copy(f"{CKPT_DIR}/admt_15k.pt", f"{CKPT_DIR}/admt_suite/last.pt")
run("train.py", "--data", DATA_DIR, "--out", f"{CKPT_DIR}/admt_suite", "--steps", 24_000, gpu_only=True)
print("ADM-T continué : pas", ckpt_step(f"{CKPT_DIR}/admt_suite/last.pt"))

In [ ]:
src = f"{CKPT_DIR}/admt_suite/last.pt"
step = ckpt_step(src)
freeze(src, f"{CKPT_DIR}/admt_suite_{step // 1000}k.pt")


## 6. Ablations
Même protocole que TDN (`TDN_ARGS` dans `config.py` : depuis ADM-T 15k, 9 000 pas), un seul réglage modifié à chaque fois (`ABLATIONS` dans `config.py`). Chaque variante coûte autant qu'un entraînement TDN : choisir lesquelles lancer avec `TO_RUN`. Une variante déjà terminée n'est pas relancée (`train.py` retrouve `last.pt` et s'arrête aussitôt) ; une variante interrompue reprend où elle en était.

`ablations_limites.ipynb` évalue celles dont le checkpoint existe.

In [ ]:
print({key: label for key, (label, _) in ABLATIONS.items()})
TO_RUN = []  # list(ABLATIONS) pour tout lancer ; ex. ["preserve", "denoise"]

for key in TO_RUN:
    label, extra_args = ABLATIONS[key]
    print(f"===== {key} : {label} =====")
    run("train.py", "--data", DATA_DIR, "--out", f"{CKPT_DIR}/abl/{key}", "--resume", f"{CKPT_DIR}/admt_15k.pt",
        *TDN_ARGS, *extra_args, gpu_only=True)